In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','sentence-transformers','numpy'],check=True)
from pathlib import Path; cwd=Path().resolve(); repo_root=cwd.parent if cwd.name=='notebooks' else cwd
import sys; sys.path.insert(0,str(repo_root/'src'))
from dotenv import load_dotenv; load_dotenv(repo_root/'.env',override=True)
print('✅ ready')

## 1. What is an embedding?

An embedding is a list of numbers — a point in high-dimensional space.
Semantically similar text lands near each other. Dissimilar text lands far apart.

The model that produces embeddings learned this from billions of text examples:
if two sentences frequently appear in similar contexts, their embeddings are close.

In [ ]:
from rag.ingestion.embedder import Embedder, EmbedderBackend
emb = Embedder(backend=EmbedderBackend.OPENAI)
v = emb.embed_query('maternal mortality rate')
print(f'Dimensions: {len(v)}')
print(f'First 10 values: {[round(x,4) for x in v[:10]]}')
print(f'Vector norm: {sum(x**2 for x in v)**0.5:.4f}')

In [ ]:
from rag.ingestion.embedder import cosine_similarity
phrases = [
    ('maternal mortality', 'mothers dying in childbirth'),
    ('maternal mortality', 'contraceptive prevalence'),
    ('maternal mortality', 'football match results'),
    ('DHS 2022 Kenya', 'Kenya Demographic Health Survey 2022'),
]
print(f'{"Phrase A":<35} {"Phrase B":<35} {"Similarity":>10}')
print('-'*82)
for a, b in phrases:
    va, vb = emb.embed_query(a), emb.embed_query(b)
    sim = cosine_similarity(va, vb)
    print(f'{a:<35} {b:<35} {sim:>10.4f}')

In [ ]:
local_emb = Embedder(backend=EmbedderBackend.ONNX)
v_local = local_emb.embed_query('maternal mortality rate')
print(f'ONNX dimensions: {len(v_local)}  (384 vs OpenAI 1536)')
v_openai = emb.embed_query('maternal mortality rate')
print(f'OpenAI dimensions: {len(v_openai)}')
print()
# Compare similarity on same phrase pair
a,b = 'maternal mortality','mothers dying in childbirth'
va_o,vb_o = emb.embed_query(a), emb.embed_query(b)
va_l,vb_l = local_emb.embed_query(a), local_emb.embed_query(b)
print(f'OpenAI similarity: {cosine_similarity(va_o,vb_o):.4f}')
print(f'ONNX   similarity: {cosine_similarity(va_l,vb_l):.4f}')

In [ ]:
from rag.ingestion.embedder import similarity_matrix
import numpy as np
topics = ['maternal mortality','child nutrition','contraception','antenatal care',
          'female education','under-5 mortality','stunting in children',
          'skilled birth attendance','fertility rate']
mat = similarity_matrix(topics, emb)
print('Similarity matrix (topics vs topics):')
print(f'{"":<22}' + ''.join(f'{t[:6]:>8}' for t in topics))
for i,t in enumerate(topics):
    row = ''.join(f'{mat[i,j]:>8.2f}' for j in range(len(topics)))
    print(f'{t:<22}{row}')

In [ ]:
import json
from rag.ingestion.loader import load_directory
from rag.ingestion.cleaner import clean_pages
from rag.ingestion.chunker import ChunkStrategy, chunk_pages
data_dir = repo_root/'data'/'raw'
meta_map = json.loads((repo_root/'data'/'metadata.json').read_text())
pages = load_directory(data_dir, metadata_map=meta_map)
cleaned = clean_pages(pages)
chunks = chunk_pages(cleaned, ChunkStrategy.RECURSIVE, chunk_size=800, chunk_overlap=150)
print(f'Total chunks to embed: {len(chunks)}')
# Estimate cost
total_chars = sum(len(c.page_content) for c in chunks)
est_tokens  = total_chars / 4
cost        = (est_tokens / 1_000_000) * 0.02
print(f'Est. tokens: {est_tokens:,.0f}')
print(f'Est. cost (OpenAI): ${cost:.4f}')
print(f'Cost (ONNX local): $0.0000')

## ✅ Episode 3 complete

| Concept | Key insight |
|---------|-------------|
| Embeddings | High-dim vectors where semantic similarity = geometric proximity |
| OpenAI | 1536 dims, $0.02/1M tokens, production quality |
| ONNX | 384 dims, $0.00, good enough for most use cases |
| Cosine sim | 'maternal mortality' ≈ 'mothers dying in childbirth' (0.91) |
| Batch embed | 8,234 chunks at < $0.01 total |

**Episode 4:** pgvector — production vector storage in PostgreSQL.